<a href="https://colab.research.google.com/github/rohit-jain8/Problem-Set-3/blob/main/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [30]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [31]:
# Create 3,000 customers
customer_ids = np.arange(1, N_CUSTOMERS + 1)

# Assign 2,100 customers to West and 900 to East
regions = np.array(["West"] * 2100 + ["East"] * 900)
rng.shuffle(regions)

# Generate the number of orders per customer
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, N_CUSTOMERS)

# Generate customer revenue
revenue = 60 * n_orders + rng.normal(400, 150, N_CUSTOMERS)
revenue = np.maximum(revenue, 50)

# Create signup dates
signup_dates = pd.date_range("2024-01-01", periods=N_CUSTOMERS, freq="D")

# Create the customers table
customers = pd.DataFrame({
    "customer_id": customer_ids,
    "region": regions,
    "revenue": revenue,
    "signup_date": signup_dates
})

# Create the orders table
orders = pd.DataFrame({
    "customer_id": np.repeat(customer_ids, n_orders)
})

orders["order_id"] = np.arange(1, len(orders) + 1)

# Register both tables in DuckDB
con.register("customers", customers)
con.register("orders", orders)

# Confirm the tables work
print(con.execute("SELECT COUNT(*) FROM customers").fetchall())
print(con.execute("SELECT COUNT(*) FROM orders").fetchall())

[(3000,)]
[(20493,)]


### Step 1.2: Write both queries (7 pts)

In [32]:
# Average revenue per West customer
query1 = """
SELECT
    AVG(revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers
WHERE region = 'West'
"""

con.execute(query1).df()

,avg_revenue,row_count
0,810.654,2100


In [33]:
# Average revenue after joining customers with orders
query2 = """
SELECT
    AVG(c.revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

con.execute(query2).df()

,avg_revenue,row_count
0,1246.412,14440


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

The correct average revenue per customer in the West region is 810.65, based on 2,100 customers. However, when we join the customers and orders tables, the average increases to 1,246.41 across 14,440 rows. This happens because customers with more orders appear multiple times in the joined table, meaning their revenue is counted more than once. Since customers with more orders also tend to generate more revenue, this causes the average to be higher than it should be. Therefore, the correct average is 810.65 because we are calculating revenue per customer, meaning each customer should only be counted once.

### Step 1.3: The rule (10 pts)

In [34]:
# Function to check whether a query changes the row count
def check_fanout(original_count, new_count):
    if new_count > original_count:
        print("WARNING: Fan-out detected!")
        print("Original rows:", original_count)
        print("New rows:", new_count)
    else:
        print("No fan-out detected.")

# Get the number of West customers
original_count = con.execute("""
    SELECT COUNT(*)
    FROM customers
    WHERE region = 'West'
""").fetchone()[0]

# Check the good query
print("Good query:")
check_fanout(original_count, original_count)

# Check the bad query
joined_count = con.execute("""
    SELECT COUNT(*)
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
""").fetchone()[0]

print("\nBad query:")
check_fanout(original_count, joined_count)

Good query:
No fan-out detected.

Bad query:
Original rows: 2100
New rows: 14440


To prevent this issue from happening again, I created a function that checks whether the number of rows increases after a join. When I tested the function on the correct query, no warning appeared because the row count stayed the same. However, when I tested it on the incorrect query, the function detected that the row count increased from 2,100 to 14,440 and displayed a warning. This check would help identify joins that duplicate customer records before incorrect averages are reported.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [35]:
# Reset the original regions
customers["region"] = regions.copy()

# Randomly select 15% of customers
missing_rows = rng.choice(
    N_CUSTOMERS,
    size=int(N_CUSTOMERS * 0.15),
    replace=False
)

# Set their region to missing
customers.loc[missing_rows, "region"] = None

# Register the updated table
con.register("customers", customers)

# Check the number of missing regions
con.execute("""
SELECT COUNT(*) AS missing_regions
FROM customers
WHERE region IS NULL
""").df()

,missing_regions
0,450


### Step 2.2: Lose them, then find them (15 pts)

In [36]:
# Count customers who are not in the West region
query3 = """
SELECT COUNT(*) AS customer_count
FROM customers
WHERE region != 'West'
"""

con.execute(query3).df()

,customer_count
0,758


In [37]:
# Count customers who are not in West, including missing regions
query4 = """
SELECT COUNT(*) AS customer_count
FROM customers
WHERE region != 'West'
   OR region IS NULL
"""

con.execute(query4).df()

,customer_count
0,1208


In [38]:
# Compare the two results
original_count = con.execute(query3).fetchone()[0]
corrected_count = con.execute(query4).fetchone()[0]

print("Original query:", original_count)
print("Corrected query:", corrected_count)
print("Missing customers:", corrected_count - original_count)

Original query: 758
Corrected query: 1208
Missing customers: 450


The original query returned 781 customers, while the corrected query returned 1,231 customers, a difference of 450. This happens because SQL does not include NULL values when using `!= 'West'`, since it cannot determine whether an unknown region is different from West. By adding `OR region IS NULL`, we include those missing customers in the results.

If customers with missing regions are systematically different, such as being newer, smaller, or more likely to leave the company, excluding them could create selection bias. This could make regional comparisons misleading because the analysis would only include customers whose regions are known, rather than accounting for the entire customer base.

### Step 2.3: Aggregate without collapsing (10 pts)

In [39]:
# Show each customer's revenue and their region's average
query5 = """
SELECT
    customer_id,
    region,
    revenue,
    AVG(revenue) OVER (PARTITION BY region) AS region_avg_revenue
FROM customers
"""

result = con.execute(query5).df()
result.head(10)

,customer_id,region,revenue,region_avg_revenue
0,1441,None,1011.346,813.732
1,2159,None,798.501,813.732
2,763,None,489.144,813.732
3,767,None,203.032,813.732
4,30,None,441.374,813.732
5,33,None,578.711,813.732
6,38,None,1432.050,813.732
7,63,None,729.419,813.732
8,69,None,1691.777,813.732
9,74,None,628.812,813.732


In [40]:
# Verify that the window function did not change the row count
print("Original rows:", len(customers))
print("Rows after window function:", len(result))

Original rows: 3000
Rows after window function: 3000


I used a window function to calculate the average revenue for each region while keeping each customer's individual revenue in the table. The `PARTITION BY region` statement calculates a separate average for each region without combining customers into a single row. For example, the West region's average revenue was 812.20, which allows us to compare each West customer's revenue against that average. The number of rows remained at 3,000 before and after the query, confirming that no customer records were removed or duplicated.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [41]:
# Create a separate random generator
rng3 = np.random.default_rng(42)

# Simulate 3,000 customers
n = 3000

# Generate existing account sizes
account_size = rng3.normal(100, 25, n)

# Larger accounts are more likely to be assigned to West
west = (account_size + rng3.normal(0, 20, n) > 100).astype(int)

# Set the true programme effect
true_effect = 20

# Generate revenue
revenue3 = 10 * account_size + true_effect * west + rng3.normal(0, 100, n)

# Create a DataFrame
causal_data = pd.DataFrame({
    "account_size": account_size,
    "west": west,
    "revenue": revenue3
})

causal_data.head()

,account_size,west,revenue
0,107.618,1,1118.691
1,74.000,0,734.670
2,118.761,1,1294.017
3,123.514,0,1295.814
4,51.224,0,213.659


In [42]:
# Compare average revenue without controlling for account size
naive_result = causal_data.groupby("west")["revenue"].mean()

naive_effect = naive_result[1] - naive_result[0]

print("Naive estimated effect:", round(naive_effect, 2))
print("True programme effect:", true_effect)

Naive estimated effect: 335.45
True programme effect: 20


In [43]:
# Register the simulated data in DuckDB
con.register("causal_data", causal_data)

# Calculate the adjusted effect using linear regression
adjusted_result = con.execute("""
SELECT
    regr_slope(revenue - 10 * account_size, west) AS adjusted_effect
FROM causal_data
""").df()

print("Naive estimated effect:", round(naive_effect, 2))
print("Adjusted estimated effect:", round(adjusted_result["adjusted_effect"][0], 2))
print("True programme effect:", true_effect)

Naive estimated effect: 335.45
Adjusted estimated effect: 22.57
True programme effect: 20


The naive comparison estimated that the account-management programme increased revenue by 335.45, even though the true effect was only 20. This happened because customers with larger accounts were more likely to be assigned to the West region, and larger accounts also tend to generate more revenue. As a result, the naive comparison attributed some of the revenue differences caused by account size to the programme itself.

After accounting for existing account size, the estimated effect decreased to 22.57, which is much closer to the true effect of 20. This demonstrates how controlling for a confounding variable can help produce a more accurate estimate of a programme's actual impact.

### Step 3.2: The control that ruins it (13 pts)

In [44]:
# Generate support tickets influenced by the programme and revenue
causal_data["support_tickets"] = (
    5 * causal_data["west"]
    + 0.01 * causal_data["revenue"]
    + rng3.normal(0, 2, n)
)

# View the updated data
causal_data.head()

,account_size,west,revenue,support_tickets
0,107.618,1,1118.691,16.393
1,74.000,0,734.670,5.110
2,118.761,1,1294.017,18.142
3,123.514,0,1295.814,10.666
4,51.224,0,213.659,4.510


In [45]:
# Estimate the effect without controlling for support tickets
effect_without_tickets = con.execute("""
SELECT regr_slope(revenue - 10 * account_size, west)
FROM causal_data
""").fetchone()[0]

# Estimate the effect after controlling for support tickets
# Register the updated data first
con.register("causal_data", causal_data)

# Remove the known account-size component
causal_data["adjusted_revenue"] = (
    causal_data["revenue"] - 10 * causal_data["account_size"]
)

# Use a simple regression with two explanatory variables
X = np.column_stack([
    np.ones(n),
    causal_data["west"],
    causal_data["support_tickets"]
])

y = causal_data["adjusted_revenue"].to_numpy()

coefficients = np.linalg.lstsq(X, y, rcond=None)[0]

effect_with_tickets = coefficients[1]

print("True programme effect:", true_effect)
print("Estimate without tickets:", round(effect_without_tickets, 2))
print("Estimate with tickets:", round(effect_with_tickets, 2))

True programme effect: 20
Estimate without tickets: 22.57
Estimate with tickets: -72.47


Although controlling for support tickets might initially seem reasonable, doing so actually makes our estimate less accurate. This is because the number of support tickets is influenced by both the account-management programme and customer revenue. By controlling for support tickets, we introduce a misleading relationship between the programme and revenue that was not present before. In our simulation, the estimated effect changed from 22.57 to -72.47 after including support tickets, even though the true effect was positive 20. This demonstrates that adding more control variables does not necessarily improve an analysis, especially when those variables are influenced by the programme or the outcome we are trying to measure.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [46]:
import re

def review_query(con, sql):
    warnings = []

    # Make the SQL lowercase and remove extra spaces
    clean_sql = " ".join(sql.lower().split())

    # Only review SELECT queries
    if not clean_sql.startswith("select"):
        return [{
            "type": "unsupported",
            "message": "Only SELECT queries are supported."
        }]

    # Check whether the query joins customers and orders
    has_join = bool(re.search(r"\bjoin\s+orders\b", clean_sql))
    has_customers = bool(re.search(r"\bfrom\s+customers\b", clean_sql))

    # Check whether the query calculates an aggregate
    has_aggregate = bool(re.search(
        r"\b(avg|sum|count)\s*\(", clean_sql
    ))

    # Check whether a region filter excludes missing values
    if re.search(r"\bregion\s*(?:!=|<>)\s*'west'", clean_sql):

        missing_count = con.execute("""
            SELECT COUNT(*)
            FROM customers
            WHERE region IS NULL
        """).fetchone()[0]

        # SQL treats comparisons with NULL as unknown,
        # so WHERE region != 'West' excludes NULL rows.
        if missing_count > 0 and not re.search(
            r"\bregion\s+is\s+null\b", clean_sql
        ):
            warnings.append({
                "type": "null_filter",
                "message": "The region filter excludes unknown values.",
                "missing_rows": missing_count
            })

    # Check whether the join increases the number of rows
    if has_join and has_customers:

        # Count the joined rows directly rather than running
        # the full original join query twice.
        joined_count = con.execute("""
            SELECT COUNT(*)
            FROM customers c
            JOIN orders o
                ON c.customer_id = o.customer_id
        """).fetchone()[0]

        customer_count = con.execute("""
            SELECT COUNT(*)
            FROM customers
        """).fetchone()[0]

        if joined_count > customer_count:
            warnings.append({
                "type": "fan_out",
                "message": "The join increases the number of rows.",
                "original_rows": customer_count,
                "joined_rows": joined_count
            })

            # Aggregates can be misleading when rows are duplicated
            if has_aggregate:
                warnings.append({
                    "type": "aggregate_fanout",
                    "message": "An aggregate is calculated after a fan-out."
                })

    return warnings

In [47]:
# Run review_query on your Phase 1 and Phase 2 queries.

# Test the reviewer on our earlier SQL queries
queries = {
    "Phase 1 - Good query": query1,
    "Phase 1 - Bad query": query2,
    "Phase 2 - Original filter": query3,
    "Phase 2 - Corrected filter": query4,
    "Phase 2 - Window function": query5
}

for name, sql in queries.items():
    print("\n" + name)

    warnings = review_query(con, sql)

    if len(warnings) == 0:
        print("No warnings found.")
    else:
        for warning in warnings:
            print(warning)


Phase 1 - Good query
No warnings found.

Phase 1 - Bad query
{'type': 'fan_out', 'message': 'The join increases the number of rows.', 'original_rows': 3000, 'joined_rows': 20493}
{'type': 'aggregate_fanout', 'message': 'An aggregate is calculated after a fan-out.'}

Phase 2 - Original filter
{'type': 'null_filter', 'message': 'The region filter excludes unknown values.', 'missing_rows': 450}

Phase 2 - Corrected filter
No warnings found.

Phase 2 - Window function
No warnings found.


The SQL reviewer successfully identified the main issues from Phases 1 and 2. It detected that the incorrect join increased the row count from 3,000 to 20,493 and warned that calculating an aggregate after this join could produce misleading results. It also identified that the original region filter excluded 450 customers with missing region values. Meanwhile, the correct queries produced no warnings.

The reviewer would have caught the issue behind the approximately 1,240 revenue figure before it reached the board. However, it has some limitations. It only checks for specific SQL patterns involving the customers and orders tables, so it may miss problems involving other tables, joins, or more complicated queries. It also checks the overall row count rather than the exact rows included in each filtered query. Therefore, while it is useful for identifying common mistakes, it should not replace manually verifying important results.

### Task 4.2: The one-page writeup (8 pts)

**To: Director**

After reviewing the revenue data, I found that the correct average revenue per customer in the West region is 810.65, rather than 1,246.41. The difference comes from how the two SQL queries calculate the average. The incorrect query joins the customers and orders tables, causing customers with more orders to appear multiple times. Since these customers also tend to generate more revenue, their revenue is given more weight, making the average higher than it should be. The correct query avoids this issue by counting each customer only once.

As for the account-management programme, our simulation suggests that its actual impact on revenue is relatively small. The initial comparison estimated an increase of 335.45, but after accounting for existing account size, this estimate decreased to 22.57, much closer to the true effect of 20. This shows that much of the original difference was due to larger customers being more likely to receive the programme, rather than the programme itself. However, we cannot assume the programme would have the same effect in reality based on this simulation alone.

Moving forward, I recommend checking row counts before and after joins, making sure missing values are handled correctly, and carefully considering which variables we control for when evaluating causal relationships. Using the SQL reviewer could also help catch potential mistakes before any results are included in presentations.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* GPT-6 and November 1, 2026

*Prompt:* I need help creating a Python function called `review_query(con, sql)` that reviews SQL queries and identifies potential mistakes before the results are used in a dashboard.

The function should work with DuckDB and return a list of warnings for three main issues:

1. Joins that increase the number of rows.
2. Filters that accidentally exclude customers with missing values.
3. Averages or other calculations that might be incorrect because a join duplicated rows.

Please use only simple Python concepts and things I would have learned in class, along with DuckDB and pandas. My tables are called `customers` and `orders`, so please use those in the code.

Also, please include comments explaining how the function identifies joins that increase row counts without running the full join twice, and why SQL can accidentally exclude missing values.

Finally, please help me test the function on my earlier queries from Phases 1 and 2, including the correct and incorrect revenue queries and the queries involving missing region values. I also want to understand what the function successfully catches, what it might miss, and whether it would have caught the incorrect revenue figure of approximately 1,240 before it was reported.

*Reply:* Absolutely! We'll create a simple SQL reviewer using Python and DuckDB, then test it on the queries from Phases 1 and 2.

The goal is to catch mistakes that could lead to incorrect results before those numbers are used in a dashboard.
Step 1: Create the review_query() function
Copy the following code into a new Google Colab code cell.

import re

def review_query(con, sql):
    warnings = []

    # Make the SQL lowercase and remove extra spaces
    clean_sql = " ".join(sql.lower().split())

    # Only review SELECT queries
    if not clean_sql.startswith("select"):
        return [{
            "type": "unsupported",
            "message": "Only SELECT queries are supported."
        }]

    # Check whether the query joins customers and orders
    has_join = bool(re.search(r"\bjoin\s+orders\b", clean_sql))
    has_customers = bool(re.search(r"\bfrom\s+customers\b", clean_sql))

    # Check whether the query calculates an aggregate
    has_aggregate = bool(re.search(
        r"\b(avg|sum|count)\s*\(", clean_sql
    ))

    # Check whether a region filter excludes missing values
    if re.search(r"\bregion\s*(?:!=|<>)\s*'west'", clean_sql):

        missing_count = con.execute("""
            SELECT COUNT(*)
            FROM customers
            WHERE region IS NULL
        """).fetchone()[0]

        # SQL treats comparisons with NULL as unknown,
        # so WHERE region != 'West' excludes NULL rows.
        if missing_count > 0 and not re.search(
            r"\bregion\s+is\s+null\b", clean_sql
        ):
            warnings.append({
                "type": "null_filter",
                "message": "The region filter excludes unknown values.",
                "missing_rows": missing_count
            })

    # Check whether the join increases the number of rows
    if has_join and has_customers:

        # Count the joined rows directly rather than running
        # the full original join query twice.
        joined_count = con.execute("""
            SELECT COUNT(*)
            FROM customers c
            JOIN orders o
                ON c.customer_id = o.customer_id
        """).fetchone()[0]

        customer_count = con.execute("""
            SELECT COUNT(*)
            FROM customers
        """).fetchone()[0]

        if joined_count > customer_count:
            warnings.append({
                "type": "fan_out",
                "message": "The join increases the number of rows.",
                "original_rows": customer_count,
                "joined_rows": joined_count
            })

            # Aggregates can be misleading when rows are duplicated
            if has_aggregate:
                warnings.append({
                    "type": "aggregate_fanout",
                    "message": "An aggregate is calculated after a fan-out."
                })

    return warnings

How the function works
The function starts with an empty list called warnings. As it reviews a query, it adds a warning whenever it identifies a potential issue.

It uses three main checks:

1. Missing values: It looks for a filter such as region != 'West'. SQL treats comparisons involving NULL as unknown, so those customers are excluded unless we explicitly include them with IS NULL. This is a correctness issue because it can leave customers out of the analysis without the analyst realizing it.

2. Fan-out: It compares the number of customers with the number of rows produced by joining customers and orders. Rather than executing the full original query twice, it uses separate COUNT(*) queries to check whether the join creates additional rows.

3. Aggregates: If the reviewer detects both a fan-out and an aggregate such as AVG(), it warns that the calculation could be misleading because some customer records may be counted multiple times.

We also use re.search() to identify SQL patterns. This is a standard Python function for finding specific text patterns.
Step 2: Test the function on our earlier queries
We already created five SQL queries in Phases 1 and 2, so we can reuse them.

Add another code cell:

# Test the reviewer on our earlier SQL queries
queries = {
    "Phase 1 - Good query": query1,
    "Phase 1 - Bad query": query2,
    "Phase 2 - Original filter": query3,
    "Phase 2 - Corrected filter": query4,
    "Phase 2 - Window function": query5
}

for name, sql in queries.items():
    print("\n" + name)

    warnings = review_query(con, sql)

    if len(warnings) == 0:
        print("No warnings found.")
    else:
        for warning in warnings:
            print(warning)

Expected results
Based on our earlier work, we should see the following:
Query	Expected warning
Phase 1 — Good query	No warnings
Phase 1 — Bad query	Fan-out and aggregate warnings
Phase 2 — Original filter	Missing-value warning
Phase 2 — Corrected filter	No warnings
Phase 2 — Window function	No warnings

For the incorrect Phase 1 query, the reviewer should identify that joining the two tables increases the row count from 3,000 to 20,493.

For the original Phase 2 filter, it should identify the 450 customers with missing region values.
Step 3: What does the reviewer catch, and what does it miss?
Our reviewer is useful for identifying the specific mistakes we encountered, but it isn't designed to detect every possible SQL error.

What it catches: It detects the problematic join between customers and orders, warns about aggregates calculated after that join, and identifies the missing values excluded by our original region filter.

What it misses: It only recognizes certain SQL patterns and table names. For example, it might miss a join written in a different order, a more complicated query, or a filter on a different nullable column. It also checks the overall row counts rather than the exact filtered rows used in every query. Additionally, not every aggregate after a join is necessarily incorrect.

These limitations are important because a reviewer should help analysts identify potential mistakes, rather than automatically determine whether every result is correct.
Step 4: Would it have caught the incorrect revenue figure?
Yes, for the specific query used in our assignment.

Our original results were:
Measurement	Revenue
Correct average per West customer	810.65
Incorrect average after joining orders	1,246.41

The incorrect query gave customers with more orders greater weight in the average. Because those customers also tended to generate more revenue, the result was inflated.

Our reviewer would flag the join and warn that an aggregate was being calculated over duplicated customer rows. It would not automatically calculate the correct revenue figure, but it would alert the analyst to investigate the query before reporting the result.
Step 5: Written evaluation for your notebook
Here's a short explanation you can include underneath the test results.

The SQL reviewer successfully identified the main issues from Phases 1 and 2. It detected that the incorrect join increased the row count from 3,000 to 20,493 and warned that calculating an aggregate after this join could produce misleading results. It also identified that the original region filter excluded 450 customers with missing region values. Meanwhile, the correct queries produced no warnings.

The reviewer would have caught the issue behind the approximately 1,240 revenue figure before it reached the board. However, it has some limitations. It only checks for specific SQL patterns involving the customers and orders tables, so it may miss problems involving other tables, joins, or more complicated queries. It also checks the overall row count rather than the exact rows included in each filtered query. Therefore, while it is useful for identifying common mistakes, it should not replace manually verifying important results.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.